# Incremental Data Loading using Auto Loader

In [0]:
%sql
CREATE SCHEMA netflix_adb_rayn.net_schema;

---------------------------------------------------------------------------
AnalysisException                         Traceback (most recent call last)
File <command-8186821662262944>, line 1
----> 1 get_ipython().run_cell_magic('sql', '', 'CREATE SCHEMA netflix_adb_rayn.net_schema;\n')

File /databricks/python/lib/python3.12/site-packages/IPython/core/interactiveshell.py:2541, in InteractiveShell.run_cell_magic(self, magic_name, line, cell)
   2539 with self.builtin_trap:
   2540     args = (magic_arg_s, cell)
-> 2541     result = fn(*args, **kwargs)
   2543 # The code below prevents the output from being displayed
   2544 # when using magics with decorator @output_can_be_silenced
   2545 # when the last Python token in the expression is a ';'.
   2546 if getattr(fn, magic.MAGIC_OUTPUT_CAN_BE_SILENCED, False):

File /databricks/python_shell/lib/dbruntime/sql_magic/sql_magic.py:214, in SqlMagic.sql(self, line, cell)
    207 except BaseException as e:
    208     self.driver_activity_lo

In [0]:
schema_loc = "abfss://silver@netflixprojectdlrayyan.dfs.core.windows.net/checkpoints/schema"
display_checkpoint = "abfss://silver@netflixprojectdlrayyan.dfs.core.windows.net/checkpoints/display"
bronze_checkpoint = "abfss://silver@netflixprojectdlrayyan.dfs.core.windows.net/checkpoints/bronze_write"

In [0]:
df = spark.readStream\
    .format("cloudFiles")\
    .option("cloudFiles.format", "csv")\
    .option("cloudFiles.schemaLocation", schema_loc)\
    .load("abfss://raw@netflixprojectdlrayyan.dfs.core.windows.net")

In [0]:
display(df, checkpointLocation=display_checkpoint)

Checkpointing to abfss://silver@netflixprojectdlrayyan.dfs.core.windows.net/checkpoints/display


In [0]:
df.writeStream\
    .option("checkpointLocation", bronze_checkpoint)\
    .trigger(availableNow=True)\
    .start("abfss://bronze@netflixprojectdlrayyan.dfs.core.windows.net/netflix_titles")